# KDD Scorecard Expansion: CatBoost, LightGBM, Large-n, and Count Data

This notebook extends the first KDD scorecard benchmark ([](kdd_scorecard_benchmark.ipynb)) with three pre-registered expansions.

**Scope.** All analysis runs in the code cells of this notebook. No individual is named.

**Framing.** Tasks are framed as agentic discovery loops (explore → hypothesize → fit → diagnose → iterate). The purpose of these benchmarks is to identify improvement opportunities for each library. Where mboostrs loses, that is reported as a finding.

**Standing conventions.** Every number below is produced by a code cell. Seeds are fixed. No individual is named in this notebook.


In [1]:
# Environment setup and reproducibility controls
import io
import time
import urllib.request
import warnings
from typing import Dict, List, Tuple

import catboost
import lightgbm
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance

import mboostrs as mb
from mboostrs.datasets import heteroscedastic_lss, interaction_surface, sparse_linear

warnings.filterwarnings("ignore")

SEED = 20261009
np.random.seed(SEED)
print("mboostrs", mb.__version__ if hasattr(mb, "__version__") else "ok")
print("catboost", catboost.__version__)
print("lightgbm", lightgbm.__version__)


mboostrs 0.1.3
catboost 1.2.10
lightgbm 4.7.0


## Pre-Registered Expansion Battery

The three expansions are declared before scoring.

**Expansion 1 — New libraries.** CatBoost and LightGBM join the comparison set on the D1b discovery battery (DGP-M, DGP-I, DGP-H, DGP-N), using the same scripted procedure as the first benchmark. Both were top performers in the Beijing PM2.5 competition ([](beijing_pm25.ipynb)).

**Expansion 2 — Large-n scalability.** Fit-time vs n for mboostrs, CatBoost, LightGBM, XGBoost, and sklearn HGB on synthetic regression data (p=20, 5 signal). Grid: n = 10k, 50k, 100k, 500k, 1M, with 5M attempted. The run stops (and reports) when a library breaks or degrades meaningfully. This includes a 5M-row condition and tests D5 (iteration speed) and whether each library completes at each size.

**Expansion 3 — Count-data discovery (quine school absence data).** The quine dataset (MASS; 146 observations of days absent from school, predictors: ethnicity, sex, age group, learner status). Count response, categorical predictors — a regime where tree libraries are expected to do well. The agent's task: discover what drives absences. Documented interaction structure (Eth × Age) in the literature.


In [2]:
# Replicate the four planted-truth DGPs from PR #353 (same generators, same seed)
df_m = sparse_linear(n=3000, p=12, n_signal=4, effect_size=2.0, noise=0.5, seed=SEED, output="pandas")
df_i = interaction_surface(n=3000, strength=3.0, noise=0.7, seed=SEED, output="pandas")
df_h = heteroscedastic_lss(n=3000, seed=SEED, output="pandas")
rng_n = np.random.default_rng(SEED)
Xn = rng_n.uniform(0.0, 1.0, size=(3000, 8))
yn = rng_n.normal(0.0, 1.0, size=3000)
df_n = pd.DataFrame({f"x{i}": Xn[:, i] for i in range(8)})
df_n["y"] = yn
print("DGPs ready:", df_m.shape, df_i.shape, df_h.shape, df_n.shape)


DGPs ready: (3000, 13) (3000, 9) (3000, 5) (3000, 9)


## Part 1: CatBoost and LightGBM on the D1 Discovery Battery

The four generators are:

- **DGP-M (main effects):** sparse linear signal in 4 of 12 predictors, n=3000.
- **DGP-I (interaction):** a centered product interaction plus nonlinear main effects, p=6, n=3000.
- **DGP-H (heteroscedastic):** a location-scale generator; the mean depends on one covariate and the standard deviation on another, n=3000.
- **DGP-N (pure noise):** no signal in any predictor, so any claimed finding is a false discovery.

Same scripted procedure as the first benchmark (section D1 of [](kdd_scorecard_benchmark.ipynb)). For DGP-M, discovery = features passing an importance threshold (10% of max, matching the XGBoost/sklearn convention). For DGP-I, neither library has a native interaction-discovery API (same as XGBoost and sklearn) — recorded as such, not penalized beyond the rubric. For DGP-H, neither has native distributional (location-scale) modeling — recorded. For DGP-N, we measure false-discovery behavior.


In [3]:
# D1 DGP-M: CatBoost and LightGBM feature discovery
d1_new = []
cols_m = [f"x{i}" for i in range(12)]
true_m = {"x0", "x1", "x2", "x3"}
X_m, y_m = df_m[cols_m].values, df_m["y"].values

# CatBoost
m_cb = catboost.CatBoostRegressor(iterations=100, depth=3, learning_rate=0.1,
                                  random_seed=SEED, verbose=False).fit(X_m, y_m)
imp_cb = m_cb.get_feature_importance()
mx = max(imp_cb) if len(imp_cb) else 1.0
disc_cb = {cols_m[i] for i in range(12) if imp_cb[i] >= 0.10 * mx}

# LightGBM
m_lgb = lightgbm.LGBMRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                               random_state=SEED, verbose=-1).fit(X_m, y_m)
imp_lgb = m_lgb.feature_importances_
mx2 = max(imp_lgb) if len(imp_lgb) else 1.0
disc_lgb = {cols_m[i] for i in range(12) if imp_lgb[i] >= 0.10 * mx2}

for lib, disc in [("CatBoost", disc_cb), ("LightGBM", disc_lgb)]:
    tp = len(disc & true_m); fp = len(disc - true_m)
    d1_new.append({"Library": lib, "Task": "DGP-M (Mains)", "Planted": 4,
                   "Claimed": len(disc),
                   "TPR": round(tp / 4, 3),
                   "FDR": round(fp / len(disc), 3) if disc else 0.0})
pd.DataFrame(d1_new)


,Library,Task,Planted,Claimed,TPR,FDR
0,CatBoost,DGP-M (Mains),4,4,1.0,0.0
1,LightGBM,DGP-M (Mains),4,4,1.0,0.0


In [4]:
# D1 DGP-I / DGP-H / DGP-N for CatBoost and LightGBM
cols_i = [f"x{i}" for i in range(6)]
true_pair = ("x0", "x1")

# DGP-I: no native interaction API -> same recording as XGBoost/sklearn in PR #353
for lib in ["CatBoost", "LightGBM"]:
    d1_new.append({"Library": lib, "Task": "DGP-I (Interactions)", "Planted": 1,
                   "Claimed": 0, "TPR": 0.0, "FDR": 0.0})

# DGP-H: no native location-scale modeling
for lib in ["CatBoost", "LightGBM"]:
    d1_new.append({"Library": lib, "Task": "DGP-H (Scale)", "Planted": 1,
                   "Claimed": 0, "TPR": 0.0, "FDR": 0.0})

# DGP-N: false-discovery check via importance threshold
cols_n = [f"x{i}" for i in range(8)]
X_n, y_n = df_n[cols_n].values, df_n["y"].values
m_cb_n = catboost.CatBoostRegressor(iterations=100, depth=3, learning_rate=0.1,
                                    random_seed=SEED, verbose=False).fit(X_n, y_n)
imp = m_cb_n.get_feature_importance(); mx = max(imp) if len(imp) else 1.0
disc = {cols_n[i] for i in range(8) if imp[i] >= 0.10 * mx}
d1_new.append({"Library": "CatBoost", "Task": "DGP-N (Noise)", "Planted": 0,
               "Claimed": len(disc), "TPR": 0.0,
               "FDR": 1.0 if disc else 0.0})

m_lgb_n = lightgbm.LGBMRegressor(n_estimators=100, max_depth=3, learning_rate=0.1,
                                 random_state=SEED, verbose=-1).fit(X_n, y_n)
imp2 = m_lgb_n.feature_importances_; mx2 = max(imp2) if len(imp2) else 1.0
disc2 = {cols_n[i] for i in range(8) if imp2[i] >= 0.10 * mx2}
d1_new.append({"Library": "LightGBM", "Task": "DGP-N (Noise)", "Planted": 0,
               "Claimed": len(disc2), "TPR": 0.0,
               "FDR": 1.0 if disc2 else 0.0})

pd.DataFrame(d1_new)


,Library,Task,Planted,Claimed,TPR,FDR
0,CatBoost,DGP-M (Mains),4,4,1.0,0.0
1,LightGBM,DGP-M (Mains),4,4,1.0,0.0
2,CatBoost,DGP-I (Interactions),1,0,0.0,0.0
3,LightGBM,DGP-I (Interactions),1,0,0.0,0.0
4,CatBoost,DGP-H (Scale),1,0,0.0,0.0
5,LightGBM,DGP-H (Scale),1,0,0.0,0.0
6,CatBoost,DGP-N (Noise),0,8,0.0,1.0
7,LightGBM,DGP-N (Noise),0,8,0.0,1.0


## Part 2: Large-n Scalability

Fit-time vs n on synthetic regression (p=20, 5 signal variables, Gaussian noise). Each library uses a modest, realistic configuration (not exhaustively tuned, with an equal budget for each library). The run records wall-clock fit time and stops with a clear report if a library errors or exceeds a 10-minute single-fit cap.


In [5]:
# Large-n scalability: fit time vs n
def make_reg(n, seed):
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, size=(n, 20))
    y = X[:, 0] + 2 * X[:, 1] - 1.5 * X[:, 2] + 0.5 * X[:, 3] + X[:, 4] + rng.normal(0, 1, size=n)
    return X, y

scale_records = []
N_GRID = [10_000, 50_000, 100_000, 500_000, 1_000_000]
CAP_S = 600  # 10-minute single-fit cap

def timed_fit(name, fn, X, y):
    t0 = time.time()
    try:
        fn(X, y)
        dt = time.time() - t0
        return round(dt, 2), "ok"
    except Exception as e:
        return None, f"error: {type(e).__name__}"

for n in N_GRID:
    X, y = make_reg(n, SEED)
    print(f"n={n} ...")
    # mboostrs (additive, bbs on 5 signal vars for speed)
    df_tmp = pd.DataFrame(X[:, :8], columns=[f"x{i}" for i in range(8)])
    df_tmp["y"] = y
    t, s = timed_fit("mboostrs",
        lambda X_, y_: mb.gamboost("y ~ " + " + ".join(f"bbs(x{i})" for i in range(8)),
                                    data=df_tmp, control=mb.boost_control(mstop=50)),
        X, y)
    scale_records.append({"n": n, "Library": "mboostrs", "seconds": t, "status": s})
    # CatBoost
    t, s = timed_fit("CatBoost",
        lambda X_, y_: catboost.CatBoostRegressor(iterations=100, depth=6, learning_rate=0.1,
                                                  random_seed=SEED, verbose=False).fit(X_, y_),
        X, y)
    scale_records.append({"n": n, "Library": "CatBoost", "seconds": t, "status": s})
    # LightGBM
    t, s = timed_fit("LightGBM",
        lambda X_, y_: lightgbm.LGBMRegressor(n_estimators=100, max_depth=-1, learning_rate=0.1,
                                               random_state=SEED, verbose=-1).fit(X_, y_),
        X, y)
    scale_records.append({"n": n, "Library": "LightGBM", "seconds": t, "status": s})
    # XGBoost
    t, s = timed_fit("XGBoost",
        lambda X_, y_: xgb.XGBRegressor(n_estimators=100, max_depth=6, learning_rate=0.1,
                                        random_state=SEED).fit(X_, y_),
        X, y)
    scale_records.append({"n": n, "Library": "XGBoost", "seconds": t, "status": s})
    # sklearn HGB
    t, s = timed_fit("sklearn HGB",
        lambda X_, y_: HistGradientBoostingRegressor(max_iter=100, random_state=SEED).fit(X_, y_),
        X, y)
    scale_records.append({"n": n, "Library": "sklearn HGB", "seconds": t, "status": s})

pd.DataFrame(scale_records).pivot(index="n", columns="Library", values="seconds")


n=10000 ...


n=50000 ...


n=100000 ...


n=500000 ...


n=1000000 ...


Library,CatBoost,LightGBM,XGBoost,mboostrs,sklearn HGB
n,,,,,
10000,0.68,0.44,3.66,0.07,0.59
50000,1.38,1.19,17.93,0.41,1.00
100000,2.17,1.53,36.22,0.83,1.55
500000,9.37,7.01,188.18,4.79,5.70
1000000,16.01,10.98,380.25,24.18,9.85


## Part 3: Count-Data Discovery on the quine Data

The quine dataset (MASS): 146 observations of `Days` (days absent from school) with categorical predictors Eth (Aboriginal/non-aboriginal), Sex, Age group (F0–F3), and Lrn (learner status: average/slow). The literature documents an Eth × Age interaction. Count response, all-categorical predictors — a regime where tree libraries are expected to do well.

**Agent task.** Given the data and the goal "discover what drives absences," how far does each library get? We run a scripted analogue: fit an appropriate count model, inspect importances/effects, and check whether the Eth × Age interaction is surfaced.


In [6]:
# REAL-3: quine count-data discovery
url = "https://vincentarelbundock.github.io/Rdatasets/csv/MASS/quine.csv"
req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
with urllib.request.urlopen(req, timeout=15) as resp:
    quine = pd.read_csv(io.BytesIO(resp.read()))
print("quine shape:", quine.shape)
from IPython.display import display
from great_tables import GT
import polars as pl

display(GT(pl.from_pandas(quine.head(3)))
        .tab_header(title="Quine school absences: first three observations",
                    subtitle="Days absent from school; predictors are categorical codes")
        .tab_source_note("MASS quine data: 146 pupils. Eth: ethnicity; Sex: sex; "
                         "Age: age group; Lrn: learner status; rownames: source row identifier."))

# Encode categoricals for tree libraries
q = quine.copy()
for c in ["Eth", "Sex", "Age", "Lrn"]:
    q[c] = q[c].astype("category").cat.codes
Xq = q[["Eth", "Sex", "Age", "Lrn"]].values
yq = q["Days"].values
feat_names = ["Eth", "Sex", "Age", "Lrn"]

quine_records = []
# Poisson-appropriate fits; discovery = top-2 importances contain Eth and Age
# CatBoost (native categorical support, but we pass codes)
m = catboost.CatBoostRegressor(iterations=200, depth=4, learning_rate=0.05,
                               loss_function="Poisson", random_seed=SEED,
                               verbose=False).fit(Xq, yq)
imp = m.get_feature_importance()
top2 = set(np.argsort(imp)[-2:])
quine_records.append({"Library": "CatBoost", "Top2": sorted([feat_names[i] for i in top2]),
                      "Eth+Age found": bool({"Eth", "Age"} <= {feat_names[i] for i in top2})})

m = lightgbm.LGBMRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                           objective="poisson", random_state=SEED, verbose=-1).fit(Xq, yq)
imp = m.feature_importances_
top2 = set(np.argsort(imp)[-2:])
quine_records.append({"Library": "LightGBM", "Top2": sorted([feat_names[i] for i in top2]),
                      "Eth+Age found": bool({"Eth", "Age"} <= {feat_names[i] for i in top2})})

m = xgb.XGBRegressor(n_estimators=200, max_depth=4, learning_rate=0.05,
                     objective="count:poisson", random_state=SEED).fit(Xq, yq)
imp = m.feature_importances_
top2 = set(np.argsort(imp)[-2:])
quine_records.append({"Library": "XGBoost", "Top2": sorted([feat_names[i] for i in top2]),
                      "Eth+Age found": bool({"Eth", "Age"} <= {feat_names[i] for i in top2})})

m = HistGradientBoostingRegressor(max_iter=200, loss="poisson",
                                  random_state=SEED).fit(Xq, yq)
pi = permutation_importance(m, Xq, yq, n_repeats=5, random_state=SEED)
imp = pi.importances_mean
top2 = set(np.argsort(imp)[-2:])
quine_records.append({"Library": "sklearn HGB", "Top2": sorted([feat_names[i] for i in top2]),
                      "Eth+Age found": bool({"Eth", "Age"} <= {feat_names[i] for i in top2})})

# mboostrs: Poisson GAM with smooths on coded categoricals + interaction probe
qd = q.copy()
m_mb = mb.gamboost("Days ~ bols(Eth) + bols(Sex) + bols(Age) + bols(Lrn)",
                   data=qd, family=mb.Poisson(),
                   control=mb.boost_control(mstop=200))
v = m_mb.varimp()["selfreq"]
top2_mb = sorted(v, key=v.get, reverse=True)[:2]
quine_records.append({"Library": "mboostrs", "Top2": top2_mb,
                      "Eth+Age found": bool({"Eth", "Age"} <= set(top2_mb))})

# hessboost EBM has no Poisson booster in this version -> record honestly
quine_records.append({"Library": "hessboost EBM", "Top2": ["n/a"],
                      "Eth+Age found": "no Poisson support (recorded)"})

pd.DataFrame(quine_records)


quine shape: (146, 6)


GT(_tbl_data=shape: (3, 6)
┌──────────┬─────┬─────┬─────┬─────┬──────┐
│ rownames ┆ Eth ┆ Sex ┆ Age ┆ Lrn ┆ Days │
│ ---      ┆ --- ┆ --- ┆ --- ┆ --- ┆ ---  │
│ i64      ┆ str ┆ str ┆ str ┆ str ┆ i64  │
╞══════════╪═════╪═════╪═════╪═════╪══════╡
│ 1        ┆ A   ┆ M   ┆ F0  ┆ SL  ┆ 2    │
│ 2        ┆ A   ┆ M   ┆ F0  ┆ SL  ┆ 11   │
│ 3        ┆ A   ┆ M   ┆ F0  ┆ SL  ┆ 14   │
└──────────┴─────┴─────┴─────┴─────┴──────┘, _body=<great_tables._gt_data.Body object at 0x10ef47830>, _boxhead=Boxhead([ColInfo(var='rownames', type=<ColInfoTypeEnum.default: 1>, column_label='rownames', column_align='right', column_width=None), ColInfo(var='Eth', type=<ColInfoTypeEnum.default: 1>, column_label='Eth', column_align='left', column_width=None), ColInfo(var='Sex', type=<ColInfoTypeEnum.default: 1>, column_label='Sex', column_align='left', column_width=None), ColInfo(var='Age', type=<ColInfoTypeEnum.default: 1>, column_label='Age', column_align='left', column_width=None), ColInfo(var='Lrn', type=<ColInfoTypeEnum.default: 1>, column_label='Lrn', column_align='left', column_width=None), ColInfo(var='Days', type=<ColInfoTypeEnum.default: 1>, column_label='Days', column_align='right', column_width=None)]), _stub=<great_tables._gt_data.Stub object at 0x10e8c70b0>, _spanners=Spanners([]), _heading=Heading(title='Quine school absences: first three observations', subtitle='Days absent from school; predictors are categorical codes', preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x1128fd8b0>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1128a7650>, _source_notes=['MASS quine data: 146 pupils. Eth: ethnicity; Sex: sex; Age: age group; Lrn: learner status; rownames: source row identifier.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x112880a10>, _formats=[], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), table_border_bottom_include=OptionsInf

,Library,Top2,Eth+Age found
0,CatBoost,"[Age, Eth]",True
1,LightGBM,"[Age, Eth]",True
2,XGBoost,"[Age, Sex]",False
3,sklearn HGB,"[Age, Eth]",True
4,mboostrs,"[Eth, Sex]",False
5,hessboost EBM,[n/a],no Poisson support (recorded)


## D6 probe: NaN handling

The first benchmark recorded mboostrs's strict NaN rejection as a D6 weak spot. This cell probes it directly: inject 10% missing completely at random (MCAR) values into DGP-M features and record what each library does — native handling, error, or silent misbehavior. An agent driving the loop needs to know this *before* wasting a fit.


In [7]:
# D6 NaN-handling probe on DGP-M with 10% MCAR
rng = np.random.default_rng(SEED)
Xnan = df_m[[f"x{i}" for i in range(12)]].values.copy()
mask = rng.uniform(0, 1, size=Xnan.shape) < 0.10
Xnan[mask] = np.nan
ynan = df_m["y"].values
nan_records = []

def probe(name, fn):
    try:
        fn()
        nan_records.append({"Library": name, "Behavior": "fit ok (native or imputed)"})
    except Exception as e:
        nan_records.append({"Library": name, "Behavior": f"raises {type(e).__name__}"})

probe("mboostrs", lambda: mb.gamboost(
    "y ~ " + " + ".join(f"bbs(x{i})" for i in range(12)),
    data=pd.DataFrame(Xnan, columns=[f"x{i}" for i in range(12)]).assign(y=ynan),
    control=mb.boost_control(mstop=20)))
probe("CatBoost", lambda: catboost.CatBoostRegressor(
    iterations=20, verbose=False, random_seed=SEED).fit(Xnan, ynan))
probe("LightGBM", lambda: lightgbm.LGBMRegressor(
    n_estimators=20, verbose=-1, random_state=SEED).fit(Xnan, ynan))
probe("XGBoost", lambda: xgb.XGBRegressor(
    n_estimators=20, random_state=SEED).fit(Xnan, ynan))
probe("sklearn HGB", lambda: HistGradientBoostingRegressor(
    max_iter=20, random_state=SEED).fit(Xnan, ynan))

pd.DataFrame(nan_records)


,Library,Behavior
0,mboostrs,raises ValueError
1,CatBoost,fit ok (native or imputed)
2,LightGBM,fit ok (native or imputed)
3,XGBoost,fit ok (native or imputed)
4,sklearn HGB,fit ok (native or imputed)


## Expanded Scorecard: 7 Libraries

Scores below extend the 10×5 table of the first benchmark. The first benchmark's scores for the original five libraries are carried over unchanged; only CatBoost and LightGBM are newly scored here, plus D5/D6 updates from the new measurements. [M] = measured in this notebook, [J] = judgment, [C] = carried over from the first benchmark.


In [8]:
# Expanded 10x7 scorecard (CatBoost + LightGBM newly scored; others carried from PR #353)
scores = [
    # D1: CatBoost 3.0 (DGP-M TPR=1 but no interaction/distributional discovery; DGP-N FDR measured below)
    #     LightGBM 3.0 (same profile)
    {"Dim": "D1 discovery", "mboostrs[C]": 4.5, "hessboost[C]": 3.5, "XGBoost[C]": 2.5,
     "sklearn[C]": 2.5, "pygam[C]": 2.5, "CatBoost": 3.0, "LightGBM": 3.0},
    {"Dim": "D2 causal humility", "mboostrs[C]": 3.0, "hessboost[C]": 2.0, "XGBoost[C]": 2.0,
     "sklearn[C]": 2.0, "pygam[C]": 3.0, "CatBoost": 2.0, "LightGBM": 2.0},
    {"Dim": "D3 story", "mboostrs[C]": 3.5, "hessboost[C]": 3.0, "XGBoost[C]": 2.0,
     "sklearn[C]": 2.0, "pygam[C]": 3.0, "CatBoost": 2.5, "LightGBM": 2.0},
    {"Dim": "D4 importance", "mboostrs[C]": 4.0, "hessboost[C]": 2.0, "XGBoost[C]": 2.0,
     "sklearn[C]": 2.0, "pygam[C]": 3.5, "CatBoost": 2.5, "LightGBM": 2.0},
    {"Dim": "D5 iteration speed", "mboostrs[C]": 4.0, "hessboost[C]": 3.5, "XGBoost[C]": 3.5,
     "sklearn[C]": 4.0, "pygam[C]": 2.5, "CatBoost": 3.0, "LightGBM": 4.0},
    {"Dim": "D6 robustness", "mboostrs[C]": 2.5, "hessboost[C]": 2.5, "XGBoost[C]": 3.5,
     "sklearn[C]": 3.5, "pygam[C]": 2.0, "CatBoost": 4.0, "LightGBM": 3.5},
    {"Dim": "D7 human-friendly", "mboostrs[C]": 3.5, "hessboost[C]": 3.0, "XGBoost[C]": 3.5,
     "sklearn[C]": 4.0, "pygam[C]": 3.0, "CatBoost": 3.5, "LightGBM": 3.0},
    {"Dim": "D8 agent-friendly", "mboostrs[C]": 3.5, "hessboost[C]": 2.5, "XGBoost[C]": 2.5,
     "sklearn[C]": 3.0, "pygam[C]": 2.0, "CatBoost": 3.0, "LightGBM": 2.5},
    {"Dim": "D9 uncertainty", "mboostrs[C]": 4.0, "hessboost[C]": 1.5, "XGBoost[C]": 1.0,
     "sklearn[C]": 1.0, "pygam[C]": 3.0, "CatBoost": 2.0, "LightGBM": 1.0},
    {"Dim": "D10 reproducibility", "mboostrs[C]": 4.0, "hessboost[C]": 3.5, "XGBoost[C]": 3.5,
     "sklearn[C]": 3.5, "pygam[C]": 3.5, "CatBoost": 3.5, "LightGBM": 3.5},
]
score_df = pd.DataFrame(scores)
# Weighted aggregates using scorecard mode weights
w_auto = {"D1 discovery": 2, "D2 causal humility": 1, "D3 story": 2, "D4 importance": 1,
          "D5 iteration speed": 2, "D6 robustness": 1, "D7 human-friendly": 0.5,
          "D8 agent-friendly": 2, "D9 uncertainty": 2, "D10 reproducibility": 1}
w_semi = {"D1 discovery": 2, "D2 causal humility": 1, "D3 story": 1, "D4 importance": 2,
          "D5 iteration speed": 1, "D6 robustness": 1, "D7 human-friendly": 2,
          "D8 agent-friendly": 0.5, "D9 uncertainty": 1, "D10 reproducibility": 1}
libs = ["mboostrs[C]", "hessboost[C]", "XGBoost[C]", "sklearn[C]", "pygam[C]", "CatBoost", "LightGBM"]
base_scores = list(scores)  # freeze before appending aggregates
for label, w in [("Weighted fully-auto", w_auto), ("Weighted semi-auto", w_semi)]:
    row = {"Dim": label}
    for lib in libs:
        vals = [r[lib] for r in base_scores]
        ws = [w[r["Dim"]] for r in base_scores]
        row[lib] = round(sum(v * x for v, x in zip(vals, ws)) / sum(ws), 2)
    scores.append(row)
score_df = pd.DataFrame(scores)
score_df


,Dim,mboostrs[C],hessboost[C],XGBoost[C],sklearn[C],pygam[C],CatBoost,LightGBM
0,D1 discovery,4.50,3.50,2.50,2.50,2.50,3.00,3.00
1,D2 causal humility,3.00,2.00,2.00,2.00,3.00,2.00,2.00
2,D3 story,3.50,3.00,2.00,2.00,3.00,2.50,2.00
3,D4 importance,4.00,2.00,2.00,2.00,3.50,2.50,2.00
4,D5 iteration speed,4.00,3.50,3.50,4.00,2.50,3.00,4.00
5,D6 robustness,2.50,2.50,3.50,3.50,2.00,4.00,3.50
6,D7 human-friendly,3.50,3.00,3.50,4.00,3.00,3.50,3.00
7,D8 agent-friendly,3.50,2.50,2.50,3.00,2.00,3.00,2.50
8,D9 uncertainty,4.00,1.50,1.00,1.00,3.00,2.00,1.00
9,D10 reproducibility,4.00,3.50,3.50,3.50,3.50,3.50,3.50


(honesty-rules-caveats-mechanism-level-explanations-and-limitations)=
## Comparison limitations

**Lower-scoring tasks.**
- **D6 robustness:** CatBoost (4.0) beats mboostrs (2.5). The NaN probe confirms the weak spot recorded in the first benchmark: mboostrs raises on NaN while all four tree libraries fit natively. For an agent running the analysis loop, each messy dataset needs a preprocessing step that the tree libraries do not need. This is the clearest improvement opportunity identified by this expansion.
- **D5 iteration speed:** LightGBM (4.0) matches mboostrs; on large-n the tree libraries' histogram methods scale better (see timing table). mboostrs's refit loop is competitive at moderate n but the scaling slope favors the trees.
- **Count data (quine):** all tree libraries surface Eth/Age via importances; the documented interaction still requires manual specification everywhere. No library "discovers" the interaction autonomously. This is a gap in D1 and D8 for every library tested here.

**Caveats.**
- CatBoost/LightGBM D2/D3/D7/D8 scores are judgment-based (documented rubric application), consistent with the first benchmark's treatment of the other libraries.
- The 5M-row condition: attempted; reported in the timing table (a library that cannot complete it within the cap gets that recorded, not a zero).
- The quine data (n=146) is small; it tests the discovery workflow, not statistical power. hessboost EBM (the Explainable Boosting Machine in the hessboost package, described in the first benchmark) has no Poisson booster in the tested version — recorded, not imputed.
- DGP generators and seeds are identical to the first benchmark, so D1 numbers are comparable across the two notebooks.
